In [14]:

from dotenv import load_dotenv
import os

from langchain_core.documents import Document

# 获取.env 中配置
env = load_dotenv()
API_KEY = os.getenv("API_KEY")
print(API_KEY)

b810c9edce884de4aada83a2cf757234.gRt9B4u7gauA2cc3


- 分析代码文件，拆分代码



In [26]:
import os


def get_file_info(rootdir):
    result = []
    # 使用 os.walk 遍历根目录及其子目录
    for root, _, files in os.walk(rootdir):
        for file in files:
            # 构建文件的完整路径
            file_path = os.path.join(root, file)
            # 获取文件名称
            file_name = file
            # 获取文件扩展名作为文件类型
            file_extension = os.path.splitext(file_name)[1].lstrip('.')
            if not file_extension:
                file_extension = 'unknown'
            try:
                # 尝试以 UTF-8 编码读取文件内容
                with open(file_path, 'r', encoding='utf-8') as f:
                    file_content = f.read()
            except UnicodeDecodeError:
                try:
                    # 若 UTF-8 解码失败，尝试以二进制模式读取并以 latin-1 编码解码
                    with open(file_path, 'rb') as f:
                        file_content = f.read().decode('latin-1', errors='replace')
                except Exception as e:
                    # 若仍有错误，记录错误信息
                    file_content = f"Error reading file: {str(e)}"
            except Exception as e:
                # 其他异常情况，记录错误信息
                file_content = f"Error reading file: {str(e)}"

            # 创建包含文件信息的字典
            file_info = {
                'file_name': file_name,
                'file_content': file_content,
                'file_type': file_extension,
                'file_path': file_path
            }
            # 将文件信息字典添加到结果列表中
            result.append(file_info)
    return result


In [30]:
def generate_analysis_prompt():
    prompt = """你是一位专业且经验丰富的人工智能助手，擅长对各类文件内容进行深入分析。接下来，请按照以下要求对指定文件进行全面剖析：
#### 一、文件基础信息
- **文件名称**：{file_name}
- **文件类型**：{file_type}
- **文件路径**：{file_path}

#### 二、文件内容分析
    {file_content}
#### 三、分析要求及输出规范

##### （一）代码内容分析（适用于 Python、Java 等编程语言代码文件）
1. **整体功能概述**
    - 简要概括代码的主要功能和预期实现的目标。

2. **类分析（如果有）**
    - 对于每个类：
        - **类名**：明确类的名称。
        - **功能用途**：详细说明该类在整个程序中的作用，它承担的任务以及与其他类的关系（如继承、实现接口等）。
        - **属性分析**：列出类的所有属性，包括属性名、数据类型、访问修饰符（如 Java 中的 `private`、`public` 等），并解释每个属性的含义和用途。
        - **方法分析**：针对类中的每个方法：
            - **方法名**：写出方法的名称。
            - **功能描述**：阐述该方法的具体功能和作用。
            - **参数列表**：列出方法的所有参数，包括参数名、数据类型、是否可选（对于 Python 可说明默认值），并解释每个参数的含义和在方法中的使用方式。
            - **返回值**：说明方法的返回值类型和返回值的含义。
            - **调用示例**：给出一个简单的调用该方法的示例代码（如果适用）。
            - **异常处理**：分析方法中是否有异常处理机制，若有，说明捕获的异常类型和处理方式。

3. **独立函数或方法分析（如果有）**
    - 对于不在类中的独立函数或方法：
        - **函数/方法名**：写出函数或方法的名称。
        - **功能描述**：说明该函数或方法的具体功能。
        - **参数列表**：列出函数或方法的所有参数，包括参数名、数据类型、是否可选（对于 Python 可说明默认值），并解释每个参数的含义和在函数或方法中的使用方式。
        - **返回值**：说明函数或方法的返回值类型和返回值的含义。
        - **调用示例**：给出一个简单的调用该函数或方法的示例代码（如果适用）。
        - **异常处理**：分析函数或方法中是否有异常处理机制，若有，说明捕获的异常类型和处理方式。

4. **代码逻辑分析**
    - 详细描述代码的执行流程，包括主要的控制结构（如循环、条件语句等）的作用和执行顺序。
    - 分析代码中使用的算法和数据结构，说明它们的选择原因和优势。

5. **代码风格和规范**
    - 评价代码的编写风格是否符合相应编程语言的最佳实践和规范。
    - 指出代码中可能存在的不规范之处或可以改进的地方。

##### （二）非代码内容分析（适用于文本、配置文件等非代码文件）
1. **内容概述**
    - 用简洁的语言概括文件内容的核心主题和主要信息。

2. **关键信息提取**
    - 提取文件中的关键数据、要点或指令，将其清晰列出。

3. **内容评估**
    - 根据文件的类型和用途，评估内容的完整性、准确性和有效性。
    - 分析内容是否存在矛盾、模糊或缺失的部分。

4. **建议与改进**
    - 针对内容评估中发现的问题，提出具体的改进建议和优化方向。
"""

    return prompt

def parseFileCodePrompt(prompt,file):
    prompt_template = PromptTemplate.from_template(prompt)
    return prompt_template.invoke({"file_name": file["file_name"],"file_type": file["file_type"],"file_path": file["file_path"],"file_content": file["file_content"]})

In [9]:
## 获取一个代码文件
## 从磁盘中读取一个代码文件
def readFile(file_path: str)->str:
    try:
        # 打开文件，'r' 表示以只读文本模式打开
        with open(file_path, 'r', encoding='utf-8') as file:
            # 读取整个文件内容
            content = file.read()
            return content
    except FileNotFoundError:
        print("指定的文件未找到。")
    except Exception as e:
        print(f"读取文件时出现错误: {e}")
## 获取代码文件后，调用大模型获取该代码文件的作用信息

from langchain_text_splitters import (
    Language,
    RecursiveCharacterTextSplitter,
)


def parseCodeFile(content: str):
    python_splitter = RecursiveCharacterTextSplitter.from_language(
        language=Language.PYTHON, chunk_size=1000, chunk_overlap=5
    )
    python_docs = python_splitter.create_documents([content])
    return python_docs


## 拆分代码文件

## embedding

## 查询








In [22]:
## 调用大模型获取代码文件的解释说明

from langchain_community.chat_models import ChatZhipuAI
from langchain_core.prompts import PromptTemplate

def parseFileCodePrompt(content: str):
    prompt_template = PromptTemplate.from_template("你是一个人工智能助手，请帮我解析以下内容，遇到代码请解释每段代码用途，入参出参等信息，请使用中文回答。 {content}")
    return prompt_template.invoke({"content": content})

def chat(message:str):
    llm = ChatZhipuAI(
        model="glm-4",
        temperature=0.8,
        api_key=API_KEY,
    )
    msg = parseFileCodePrompt(message)
    response = llm.invoke(msg)
    return response.content


In [24]:


if __name__ == '__main__':
    content = readFile(file_path="/mnt/e/project/ai/AiPy/RAG/day01/LangchainRag代码分析.ipynb")
    content_desc  = chat(content)
    print(content_desc)
    docs = parseCodeFile(content_desc)

    for doc in docs:
        print(doc)
        print("\n")





text='你是一个人工智能助手，请帮我解析以下内容，遇到代码请解释代码用途，入参出参等信息，请使用中文回答。 {\n "cells": [\n  {\n   "cell_type": "code",\n   "id": "initial_id",\n   "metadata": {\n    "collapsed": true,\n    "ExecuteTime": {\n     "end_time": "2025-02-23T10:23:10.047423Z",\n     "start_time": "2025-02-23T10:23:10.038228Z"\n    }\n   },\n   "source": [\n    "\\n",\n    "from dotenv import load_dotenv\\n",\n    "import os\\n",\n    "\\n",\n    "from langchain_core.documents import Document\\n",\n    "\\n",\n    "# 获取.env 中配置\\n",\n    "env = load_dotenv()\\n",\n    "API_KEY = os.getenv(\\"API_KEY\\")\\n",\n    "print(API_KEY)"\n   ],\n   "outputs": [\n    {\n     "name": "stdout",\n     "output_type": "stream",\n     "text": [\n      "b810c9edce884de4aada83a2cf757234.gRt9B4u7gauA2cc3\\n"\n     ]\n    }\n   ],\n   "execution_count": 14\n  },\n  {\n   "metadata": {},\n   "cell_type": "markdown",\n   "source": [\n    "- 分析代码文件，拆分代码\\n",\n    "\\n"\n   ],\n   "id": "d27bdffffc340d79"\n  },\n  {\n   "metadata": {\n    "ExecuteT

In [34]:
import os

def write_content_to_file(file_name, file_sub_path, file_content):
    # 固定的根路径
    root_path = "/mnt/e/project/ai/AiPy/RAG/day01/project_learing"
    # 拼接完整的文件路径，包含子路径
    full_path = os.path.join(root_path, file_sub_path, f"{file_name}.md")

    # 创建目录（如果不存在）
    directory = os.path.dirname(full_path)
    if not os.path.exists(directory):
        os.makedirs(directory)

    try:
        # 以写入模式打开文件
        with open(full_path, 'w', encoding='utf-8') as file:
            # 将文件内容写入文件
            file.write(file_content)
        print(f"文件 {full_path} 写入成功。")
    except Exception as e:
        print(f"写入文件时出错: {e}")

In [39]:
## 调用大模型获取代码文件的解释说明

from langchain_community.chat_models import ChatZhipuAI
from langchain_core.prompts import PromptTemplate

def chatMsg(message):
    llm = ChatZhipuAI(
        model="glm-4",
        temperature=0.8,
        api_key=API_KEY,
    )
    response = llm.invoke(message)
    return response.content

In [40]:
if __name__ == "__main__":
   files  = get_file_info("/mnt/e/project/ai/AiPy/RAG/day01/project")
   prompt = generate_analysis_prompt()
   for file in files:
       promptInfo = parseFileCodePrompt(prompt,file)
       print(promptInfo)
       content_desc  = chatMsg(promptInfo)
       ## embedding
       write_content_to_file(file["file_name"], "", content_desc)







text='你是一位专业且经验丰富的人工智能助手，擅长对各类文件内容进行深入分析。接下来，请按照以下要求对指定文件进行全面剖析：\n#### 一、文件基础信息\n- **文件名称**：new_qiangpiao.py\n- **文件类型**：py\n- **文件路径**：/mnt/e/project/ai/AiPy/RAG/day01/project/new_qiangpiao.py\n\n#### 二、文件内容分析\n    #!/usr/bin/env python\n# -*- coding: utf-8 -*-\n\n"""\n通过splinter刷12306火车票\n进入登陆页面，可以选择扫码登陆或者账号密码登陆\n登陆成功后，接下来的事情，交由脚本来做了，静静的等待抢票结果就好（刷票过程中，浏览器不可关闭）\n抢票成功，会进行手机短信和邮件的通知\nauthor: cuizy\ntime: 2018-11-21\n"""\n\nimport re\nfrom splinter.browser import Browser\nfrom time import sleep\nimport sys\nimport httplib2\nfrom urllib import parse\nimport smtplib\nfrom email.mime.text import MIMEText\nimport time\n\n\nclass BrushTicket(object):\n    """买票类及实现方法"""\n\n    def __init__(self, passengers, from_time, from_station, to_station, number, seat_type, receiver_mobile,\n                 receiver_email):\n        """定义实例属性，初始化"""\n        # 乘客姓名\n        self.passengers = passengers\n        # 起始站和终点站\n        self.from_station = from_station\n        self.to_station = to_station\n  